# IA per Tutti · Pontremoli — Modulo 2: i Token

## Cosa è un token?

Quando voi scrivete a ChatGPT o Claude, il modello **non legge le parole**. Le spezza in pezzettini più piccoli che si chiamano **token**. A volte un token è una parola intera, a volte è solo un pezzo di parola (un prefisso, una sillaba, una desinenza).

I token sono l'**unità di misura** dell'AI: quanto state mandando, quanto sta rispondendo, quanto costa, e — soprattutto — quanto **contesto** sta usando. In questi esercizi vediamo come funziona davvero.

---
**Come iniziare:** esegui prima la cella di setup qui sotto (▶), poi esegui gli esercizi uno alla volta.

In [ ]:
# @title ▶️ Setup · Esegui questa cella per iniziare { display-mode: "form" }
# @markdown Premi il bottone a sinistra per installare le librerie e preparare gli esercizi.
# @markdown Può richiedere un minuto. Quando vedi “Setup completato”, sei pronto!

!pip install tiktoken transformers -q

import tiktoken
from transformers import AutoTokenizer
import requests
import re
import html as htmllib
import matplotlib.pyplot as plt

# Tokenizer di GPT-4 / ChatGPT — usato nella maggior parte degli esercizi
enc = tiktoken.get_encoding("cl100k_base")

# Header per le chiamate a Wikipedia (richiesto dalla loro API)
WIKI_HEADERS = {"User-Agent": "IAperTutti-Pontremoli/1.0 (workshop)"}

print("✅ Setup completato — puoi eseguire gli esercizi qui sotto.")


---
## Esercizio 1 — Vedere i token con gli occhi

Prendiamo una frase semplice: `"Ciao, come stai?"`

La diamo in pasto al tokenizer e guardiamo **cosa esce fuori**. Sono numeri (gli ID dei token), e ogni numero corrisponde a un pezzettino di testo.

In [ ]:
# @title 🔤 Esercizio 1 · Vedi i token con i tuoi occhi { display-mode: "form" }
# @markdown Scrivi una frase nel campo qui sotto e premi ▶️ per vedere come viene spezzata in token.

frase = "Ciao, come stai?" # @param {type:"string"}

token_ids = enc.encode(frase)

print("📝 Frase originale:", frase)
print("🔢 Numero di token:", len(token_ids))
print("🆔 ID dei token:", token_ids)
print()
print("✨ Adesso vediamo COSA contiene ogni token:")
print("-" * 40)

for i, token_id in enumerate(token_ids):
    pezzo = enc.decode([token_id])
    print(f"  Token {i + 1:>2}  •  ID: {token_id:>6}  •  contiene: {repr(pezzo)}")


Avete visto? `"Ciao"` viene spezzato in `"C"` + `"iao"`, `"come"` resta intero (con lo spazio davanti incluso!), `"stai"` diventa `" st"` + `"ai"`.

**Lo spazio fa parte del token successivo.** È un dettaglio strano ma importante.

---
## Esercizio 2 — Come funziona davvero il tokenizer (BPE) e perché lo spazio davanti conta

I token non sono né lettere né parole — sono **pezzi di byte ricorrenti** che il tokenizer ha imparato dai dati di training. La cosa più importante (e meno intuitiva): **lo spazio davanti alla parola conta tantissimo**. Nei testi normali le parole hanno quasi sempre uno spazio davanti, quindi il tokenizer ha imparato versioni delle parole *con lo spazio incluso* — e quelle sono molto più efficienti.

Guardiamo cosa succede confrontando la stessa parola con e senza spazio, e parole comuni vs gibberish.

In [ ]:
# @title 🧩 Esercizio 2 · Lo spazio davanti cambia tutto { display-mode: "form" }
# @markdown Premi ▶️ per vedere il confronto. Guarda bene il numero di token con e senza lo spazio davanti.

esempi = [
    "Roma",
    " Roma",
    "ciao",
    " ciao",
    "xqzk",
    "internazionale",
    "Pontremoli",
]

# Calcoliamo prima tutto, poi stampiamo con colonne allineate
righe = []
for s in esempi:
    ids = enc.encode(s)
    pezzi = [enc.decode([i]) for i in ids]
    etichetta = "(con spazio)   " if s.startswith(" ") else "(senza spazio) "
    righe.append((repr(s), etichetta, len(ids), pezzi))

# Larghezza massima della colonna stringa per allineare
larghezza = max(len(r[0]) for r in righe)

for stringa, etichetta, n_token, pezzi in righe:
    pezzi_repr = "[" + ", ".join(repr(p) for p in pezzi) + "]"
    print(f"{stringa:<{larghezza}}  {etichetta} → {n_token} token: {pezzi_repr}")


**Cosa notate:**

- Lo **spazio davanti** cambia tutto: `' Roma'` è 1 token, `'Roma'` ne usa 2. Stesso per `' ciao'` vs `'ciao'`.
- `xqzk` e `ciao` hanno la **stessa lunghezza** (4 caratteri), ma `xqzk` viene spezzato in 4 token (uno per carattere) perché è gibberish — quella sotto-sequenza non l'ha mai vista.
- `internazionale` è lunga 15 caratteri ma resta in 2-3 token perché è una parola comune.
- `Pontremoli` è lunga 10 caratteri e viene spezzata in 3 token perché è rara.

**Morale:** conta la **familiarità della sotto-sequenza di byte**, non la lunghezza della parola. E lo spazio davanti è parte della sotto-sequenza.

---
## Esercizio 3 — Indovina prima: un paragrafo italiano

Adesso giochiamo a indovinare. Leggi il paragrafo qui sotto e **stima a occhio** quanti token sono.

Suggerimento: una regola spannometrica è **1 token ≈ 3-4 caratteri** in italiano. Ma è solo una stima, prova a contare!

In [ ]:
# @title 📖 Esercizio 3 · Leggi il paragrafo { display-mode: "form" }
# @markdown Leggi questo paragrafo su Pontremoli, poi vai alla cella sotto e prova a stimare quanti token è.

paragrafo = (
    "Pontremoli è il comune più a nord della Toscana, in Lunigiana. "
    "Sorge alla confluenza del fiume Magra con il torrente Verde, "
    "dominata dal Castello del Piagnaro che ospita il Museo delle Statue Stele. "
    "È tappa storica della Via Francigena e patria del Premio Bancarella. "
    "Famosa per i testaroli col pesto, i panigacci e la torta d'erbe. "
    "D'inverno si accendono i falò di San Nicolò e San Geminiano."
)

print(paragrafo)
print()
print(f"📐 Caratteri totali: {len(paragrafo)}")
print()
print("👇 Adesso passa alla cella sotto e scrivi la tua stima.")


In [ ]:
# @title 🎯 Esercizio 3 · Indovina i token del paragrafo { display-mode: "form" }
# @markdown Scrivi la tua stima qui sotto, poi premi ▶️ per scoprire il risultato.
# @markdown Suggerimento: in italiano, 1 token ≈ 3-4 caratteri.

mia_stima = 0  # @param {type:"integer"}

if mia_stima == 0:
    print("🤔 Non hai ancora messo una stima — scrivi un numero sopra e riesegui questa cella.")
else:
    token_veri = len(enc.encode(paragrafo))
    differenza = abs(token_veri - mia_stima)

    print(f"✍️ La tua stima:    {mia_stima}")
    print(f"🎯 Token veri:       {token_veri}")
    print(f"📏 Differenza:      {differenza}")
    print()
    if differenza <= 10:
        print("🏆 Ottima stima! Sei vicinissimo.")
    elif differenza <= 30:
        print("👍 Buona stima, sei nel campo da gioco.")
    else:
        print("📚 Stimare i token è difficile! Ora hai un'idea più chiara.")


---
## Quanto costano i token? Setup prezzi

Per i prossimi esercizi vediamo anche quanto costerebbe inviare ogni testo a un'AI via API. Sei modelli, sei prezzi diversi.

In [ ]:
# @title 💰 Setup · Prezzi dei provider AI { display-mode: "form" }
# @markdown Premi ▶️ per caricare i prezzi (in $ per 1M token) dei sei modelli che confronteremo.

PREZZI = {
    "Mistral Small 3":   {"input": 0.10, "output": 0.30},
    "GPT-5 Mini":        {"input": 0.25, "output": 2.00},
    "Claude Haiku 4.5":  {"input": 1.00, "output": 5.00},
    "Gemini 3.1 Pro":    {"input": 2.00, "output": 12.00},
    "Claude Sonnet 4.6": {"input": 3.00, "output": 15.00},
    "Claude Opus 4.7":   {"input": 5.00, "output": 25.00},
}

def costo(token_input, token_output, provider):
    p = PREZZI[provider]
    return (token_input * p["input"] + token_output * p["output"]) / 1_000_000

def stampa_costi(token_input, token_output=0, titolo="💰 Costo via API per questo testo"):
    print(titolo)
    for nome in PREZZI:
        c = costo(token_input, token_output, nome)
        print(f"   {nome:<22} ${c:.4f}")
    print()

print("✅ Prezzi caricati — pronti per i confronti negli esercizi.")


---
## Esercizio 4 — Indovina prima: la pagina Wikipedia di Pontremoli

Adesso prendiamo qualcosa di più lungo: **l'intera pagina Wikipedia di Pontremoli**. La scarichiamo direttamente dall'API di Wikipedia.

Prima vediamo il **primo paragrafo**, poi tu stimi quanti token c'è in tutta la pagina.

In [ ]:
# @title 📚 Esercizio 4 · Scarica la pagina Wikipedia di Pontremoli { display-mode: "form" }
# @markdown Premi ▶️ per scaricare l'intera pagina Wikipedia. Vedrai un'anteprima e i caratteri totali.

URL_WIKI = "https://it.wikipedia.org/w/api.php"

parametri = {
    "action": "query",
    "format": "json",
    "titles": "Pontremoli",
    "prop": "extracts",
    "explaintext": True,
    "exlimit": 1,
}

risposta = requests.get(URL_WIKI, params=parametri, headers=WIKI_HEADERS, timeout=15)
dati = risposta.json()

pagine = dati["query"]["pages"]
testo_wiki = list(pagine.values())[0]["extract"]

print(f"📐 Caratteri totali della pagina: {len(testo_wiki)}")
print()
print("📄 PRIMO PARAGRAFO (anteprima):")
print("-" * 40)
print(testo_wiki[:500])
print("...")
print()
print("👇 Adesso vai alla cella sotto e stima i token per TUTTA la pagina.")


In [ ]:
# @title 🎯 Esercizio 4 · Indovina i token della pagina intera { display-mode: "form" }
# @markdown Scrivi la tua stima qui sotto, poi premi ▶️ per scoprire il risultato.
# @markdown Quanti token pensi che abbia TUTTA la pagina Wikipedia (non solo l'anteprima)?

mia_stima = 0  # @param {type:"integer"}

if mia_stima == 0:
    print("🤔 Non hai ancora messo una stima — scrivi un numero sopra e riesegui questa cella.")
else:
    token_veri = len(enc.encode(testo_wiki))
    differenza = abs(token_veri - mia_stima)

    print(f"✍️ La tua stima:                {mia_stima}")
    print(f"🎯 Token veri (tutta la pagina): {token_veri}")
    print(f"📏 Differenza:                  {differenza}")
    print()
    print("💡 Per riferimento: una chat di ChatGPT/Claude ha una finestra")
    print("   di contesto di circa 200.000 token. Questa pagina ne usa solo")
    print("   una piccola parte.")
    print()
    numero_token_wikipedia = token_veri
    stampa_costi(numero_token_wikipedia, 0, "💰 Quanto costerebbe DARE questo articolo a un'AI?")
    print("   ↑ Solo l'input. Se chiedessi un riassunto, aggiungi il costo della risposta.")


---
## Esercizio 5 — Indovina prima: il primo capitolo dei Promessi Sposi

Adesso facciamo sul serio. Scarichiamo da Wikisource il **primo capitolo intero** dei Promessi Sposi di Manzoni ("Quel ramo del lago di Como..."). È un testo lungo, classico, di pubblico dominio.

Prima leggiamo l'incipit, poi tu indovini i token totali.

In [ ]:
# @title 📖 Esercizio 5 · Scarica il Capitolo I dei Promessi Sposi { display-mode: "form" }
# @markdown Premi ▶️ per scaricare il primo capitolo intero di Manzoni da Wikisource.

URL_WIKISOURCE = "https://it.wikisource.org/w/api.php"

parametri = {
    "action": "parse",
    "format": "json",
    "page": "I promessi sposi (1840)/Capitolo I",
    "prop": "text",
}

risposta = requests.get(URL_WIKISOURCE, params=parametri, headers=WIKI_HEADERS, timeout=20)
html_capitolo = risposta.json()["parse"]["text"]["*"]

html_pulito = re.sub(r"<style[^>]*>.*?</style>", "", html_capitolo, flags=re.DOTALL)
html_pulito = re.sub(r"<script[^>]*>.*?</script>", "", html_pulito, flags=re.DOTALL)
html_pulito = re.sub(r"<dc:[^>]*>.*?</dc:[^>]*>", "", html_pulito, flags=re.DOTALL)
testo_capitolo = re.sub(r"<[^>]+>", "", html_pulito)
testo_capitolo = htmllib.unescape(testo_capitolo)
testo_capitolo = re.sub(r"\[p\.\s*\d+\s*modifica\s*\]", "", testo_capitolo)
testo_capitolo = re.sub(r"\s+", " ", testo_capitolo).strip()

inizio = testo_capitolo.find("uel ramo del lago di Como")
if inizio > 0:
    testo_capitolo = "Q" + testo_capitolo[inizio:]

print(f"📐 Caratteri totali del capitolo: {len(testo_capitolo)}")
print()
print("📜 INCIPIT:")
print("-" * 40)
print(testo_capitolo[:600])
print("...")
print()
print("👇 Adesso vai alla cella sotto e stima i token per il capitolo intero.")


In [ ]:
# @title 🎯 Esercizio 5 · Indovina i token del capitolo intero { display-mode: "form" }
# @markdown Scrivi la tua stima qui sotto, poi premi ▶️ per scoprire il risultato.
# @markdown Quanti token pensi che abbia tutto il Capitolo I dei Promessi Sposi?

mia_stima = 0  # @param {type:"integer"}

if mia_stima == 0:
    print("🤔 Non hai ancora messo una stima — scrivi un numero sopra e riesegui questa cella.")
else:
    token_veri = len(enc.encode(testo_capitolo))
    differenza = abs(token_veri - mia_stima)

    print(f"✍️ La tua stima:                {mia_stima}")
    print(f"🎯 Token veri (capitolo intero): {token_veri}")
    print(f"📏 Differenza:                  {differenza}")
    print()
    print("💡 Per dare un'idea: questo è un capitolo solo. Tutto il romanzo")
    print("   sono 38 capitoli — circa 400.000-500.000 token in totale.")
    print("   NON entra in una sola chat di ChatGPT/Claude.")
    print()
    numero_token_promessi = token_veri
    stampa_costi(numero_token_promessi, 0, "💰 Quanto costerebbe DARE tutto questo capitolo a un'AI?")


---
## Esercizio 6 — La chicca: perché aprire chat nuove?

Adesso il pezzo grosso. Simuliamo una chat di **10 turni** — voi scrivete, il modello risponde, voi scrivete, il modello risponde, e così via.

**La cosa che (quasi) nessuno sa:** ogni volta che voi mandate un messaggio, il modello **non riceve solo il vostro ultimo messaggio**. Riceve **tutta la chat dall'inizio**. System prompt + messaggio 1 + risposta 1 + messaggio 2 + risposta 2 + ... + il vostro ultimo messaggio.

Quindi i token che vi vengono "contati" crescono **ad ogni turno**.

Vediamo come.

In [ ]:
# @title 💬 Esercizio 6 · Prepara la chat di 10 turni { display-mode: "form" }
# @markdown Premi ▶️ per caricare i 10 messaggi finti e le 10 risposte del modello.

# Il system prompt iniziale (presente in ogni chat)
system_prompt = (
    "Sei un assistente esperto della Lunigiana, gentile e preciso. "
    "Rispondi sempre in italiano, in modo concreto e utile."
)

# I 10 messaggi dell'utente (finti, plausibili per la nostra sala)
messaggi_utente = [
    "Ciao! Sto pianificando un weekend a Pontremoli, mi puoi aiutare?",
    "Cosa c'è da vedere assolutamente in centro?",
    "E dal punto di vista del cibo? Quali sono i piatti tipici?",
    "Dove mi consigli di mangiare i testaroli col pesto?",
    "Hai consigli per una passeggiata facile in zona, niente di troppo tecnico?",
    "E se piove? Cosa posso fare al coperto?",
    "Quanto tempo ci vuole in macchina da Milano?",
    "C'è qualche evento particolare in questo periodo dell'anno?",
    "Mi consigli un B&B carino ma non troppo caro?",
    "Perfetto, grazie! Mi fai un riassunto del piano per il weekend?",
]

# Le 10 risposte finte del modello
risposte_modello = [
    "Certo, con piacere! Pontremoli è una scelta splendida per un weekend. "
    "È il comune più a nord della Toscana, in piena Lunigiana, e ha tutto: storia, "
    "natura, e una cucina che non ti aspetti. Dimmi pure quanti giorni hai a disposizione "
    "e da dove parti, così posso darti consigli più mirati. In generale ti consiglio di "
    "alternare visite in centro storico, qualche passeggiata e ovviamente le soste "
    "gastronomiche, che sono parte essenziale dell'esperienza.",
    "Il cuore del centro è il Castello del Piagnaro, che domina il borgo dall'alto e "
    "ospita il Museo delle Statue Stele Lunigianesi, sculture in pietra di epoca "
    "preistorica davvero suggestive. Poi vale la pena perdersi tra i vicoli del centro "
    "storico, attraversare i ponti sul Magra, fermarsi in Piazza della Repubblica e "
    "visitare il Duomo. Se ami le ville nobiliari, Villa Dosi Delfini è imperdibile, "
    "con il suo parco e i due cedri del Libano del 1863.",
    "La cucina pontremolese è una delle più caratteristiche della Lunigiana. I piatti "
    "da provare assolutamente sono: i testaroli, una pasta antichissima cotta in testi "
    "di terracotta e condita col pesto pontremolese (presidio Slow Food); i panigacci, "
    "dischetti di pasta cotti tra dischi di terracotta arroventati; la torta d'erbe, "
    "una torta salata ricca di verdure di campo; la spongata, dolce natalizio; e "
    "l'Amor pontremolese, dolce inventato all'Antico Caffè degli Svizzeri.",
    "Ci sono diverse trattorie storiche che fanno testaroli eccellenti. Senza fare nomi "
    "specifici (i gusti cambiano e i locali pure), ti consiglio di cercare osterie nel "
    "centro storico o nelle frazioni vicine, magari informandoti all'ufficio turistico "
    "per avere segnalazioni aggiornate. Diffida dei posti troppo turistici: i testaroli "
    "veri si trovano dove mangiano i pontremolesi. Un consiglio: prenotare sempre, "
    "soprattutto nel weekend, perché i posti buoni si riempiono.",
    "Per una passeggiata facile ti consiglio un tratto della Via Francigena, il "
    "cammino storico che attraversa proprio Pontremoli. Puoi fare una piccola tappa "
    "verso il Passo della Cisa, oppure scendere verso sud verso Filattiera. Sono "
    "percorsi ben segnalati, non tecnici, e attraversano paesaggi bellissimi tra "
    "boschi e borghi medievali. Se preferisci qualcosa di ancora più tranquillo, "
    "la passeggiata lungo il fiume Magra dal centro è una bella alternativa.",
    "Se piove, hai comunque diverse opzioni interessanti: il Museo delle Statue Stele "
    "al Castello del Piagnaro è coperto e vale assolutamente una visita lunga. Poi c'è "
    "il Museo del Premio Bancarella, dedicato alla tradizione dei librai ambulanti "
    "pontremolesi. Le chiese del centro (Duomo, Santissima Annunziata) sono interessanti "
    "da visitare con calma. E ovviamente, una lunga sosta al ristorante con i piatti "
    "tipici trasforma una giornata di pioggia in qualcosa di memorabile.",
    "Da Milano a Pontremoli ci vogliono circa 2 ore e mezza in macchina, dipende dal "
    "traffico. Il percorso più rapido è autostrada A1 fino a Parma, poi A15 "
    "Parma-La Spezia, uscita Pontremoli. È un'uscita comoda, sei subito in centro. "
    "In treno è un po' meno comodo: la stazione di Pontremoli è sulla linea "
    "Parma-La Spezia, treni regionali, frequenza media. In macchina hai più libertà "
    "per esplorare anche i dintorni.",
    "Dipende da quando vieni. A gennaio ci sono due eventi suggestivi: la Disfida dei "
    "Falò, il 17 gennaio (San Nicolò) e il 31 gennaio (San Geminiano) — due rioni si "
    "sfidano accendendo enormi falò. Ad agosto c'è Medievalis, una rievocazione "
    "storica della visita di Federico II nel 1226: costumi d'epoca, banchetti, "
    "spettacoli. A luglio invece c'è il Premio Bancarella, manifestazione letteraria "
    "storica. Fammi sapere il periodo e ti dico se c'è qualcosa di specifico.",
    "A Pontremoli e dintorni ci sono diverse soluzioni interessanti, soprattutto "
    "agriturismi e B&B a conduzione familiare nelle frazioni intorno al centro. Per "
    "un weekend ti consiglio di cercare strutture nei borghi vicini come Filattiera, "
    "Bagnone o Mulazzo: spesso si trovano ottimi rapporti qualità-prezzo, con "
    "colazione fatta in casa e accoglienza calorosa. Booking e Airbnb hanno una buona "
    "selezione. Prenota con qualche settimana di anticipo se vieni nei weekend di alta stagione.",
    "Eccolo il tuo weekend a Pontremoli: arrivi venerdì sera, cena a base di testaroli "
    "col pesto in un'osteria del centro. Sabato mattina visita al Castello del Piagnaro "
    "e al Museo delle Statue Stele, pranzo con panigacci e torta d'erbe. Pomeriggio "
    "passeggiata su un tratto della Via Francigena. Cena tranquilla in B&B o in trattoria "
    "con specialità locali. Domenica mattina giro nel centro storico, caffè con Amor "
    "pontremolese all'Antico Caffè degli Svizzeri, e rientro nel pomeriggio. Buon viaggio!",
]

print("✅ Chat caricata: 10 messaggi dell'utente + 10 risposte del modello.")
print("👇 Adesso esegui la cella sotto per simulare la conversazione turno per turno.")


In [ ]:
# @title 🔢 Predizione 1 · Il SINGOLO turno { display-mode: "form" }
# @markdown Stiamo per simulare una chat di 10 turni. Concentriamoci su **un singolo momento**: quando tu premi INVIO sul tuo 10° messaggio, in quell'istante quanti token vengono spediti al modello (system prompt + tutta la chat fino a lì)?
# @markdown
# @markdown Scegli la tua stima e premi ▶️.

stima_turno_10 = "— scegli —"  # @param ["— scegli —", "100 token", "500 token", "1.500 token", "5.000 token", "20.000 token"]

# --- Simulazione silenziosa: calcoliamo tutto sotto il cofano ---
chat_completa = system_prompt
lista_turni = []
lista_token_messaggio = []
lista_token_risposta = []
lista_token_cumulativi = []
lista_totale_processato = []
totale_processato = 0

for turno in range(10):
    messaggio = messaggi_utente[turno]
    risposta = risposte_modello[turno]
    token_messaggio = len(enc.encode(messaggio))
    token_risposta = len(enc.encode(risposta))
    chat_completa = chat_completa + "\n" + messaggio
    token_cumulativi = len(enc.encode(chat_completa))
    totale_processato = totale_processato + token_cumulativi
    chat_completa = chat_completa + "\n" + risposta
    lista_turni.append(turno + 1)
    lista_token_messaggio.append(token_messaggio)
    lista_token_risposta.append(token_risposta)
    lista_token_cumulativi.append(token_cumulativi)
    lista_totale_processato.append(totale_processato)

contenuto_unico = len(enc.encode(chat_completa))
totale_processato_finale = lista_totale_processato[-1]
moltiplicatore = totale_processato_finale / contenuto_unico

# Valore vero al turno 10
vero_turno_10 = lista_token_cumulativi[-1]

if stima_turno_10 != "— scegli —":
    # Parse della stima — "1.500 token" -> 1500 (punto come separatore migliaia)
    parte_numerica = stima_turno_10.split(" ")[0]
    stima_int = int(parte_numerica.replace(".", ""))

    # Trova l'opzione più vicina al valore vero
    opzioni = [100, 500, 1500, 5000, 20000]
    opzione_piu_vicina = min(opzioni, key=lambda x: abs(x - vero_turno_10))

    print(f"🎯 La tua stima: {stima_int:,} token".replace(",", "."))
    print(f"✅ Risultato vero: {vero_turno_10:,} token".replace(",", "."))
    print()

    if stima_int == opzione_piu_vicina:
        print("🏆 Hai indovinato! Era la risposta più vicina.")
        print("Ogni turno il modello si trascina dietro tutta la chat — è già parecchio.")
    elif stima_int < vero_turno_10:
        print("📉 Hai sottostimato! Ogni turno trascina dietro tutta la storia:")
        print("system prompt + tutti i messaggi e risposte precedenti + il tuo nuovo messaggio.")
    else:
        print("📈 Hai sovrastimato — un singolo turno cresce, ma non così tanto.")
        print("La sorpresa vera arriva dopo: guarda la prossima predizione...")
else:
    print("🤔 Scegli una risposta dal menu sopra e riesegui ▶️")


In [ ]:
# @title 📈 Predizione 2 · La SOMMA di tutti i turni { display-mode: "form" }
# @markdown Ora alziamo la testa e guardiamo il quadro intero. Il contenuto **unico** della chat (system prompt + 10 messaggi + 10 risposte) è ~1.571 token.
# @markdown
# @markdown **Ma** ad ogni turno il modello ha riletto dall'inizio. Sommando i token che ha processato in TUTTI i 10 turni (turno 1 + turno 2 + ... + turno 10), quante volte il contenuto unico?
# @markdown
# @markdown Scegli il tuo moltiplicatore e premi ▶️.

stima_moltiplicatore = "— scegli —"  # @param ["— scegli —", "1× (uguale al contenuto)", "2×", "5×", "10×", "50×", "100×"]

def fmt(n):
    return f"{int(round(n)):,}".replace(",", ".")

if stima_moltiplicatore != "— scegli —":
    # Parse del moltiplicatore — "5×" -> 5, "1× (uguale al contenuto)" -> 1
    parte_numerica = stima_moltiplicatore.split("×")[0].strip()
    stima_molt = int(parte_numerica)

    # Le variabili contenuto_unico, totale_processato_finale, moltiplicatore sono già state
    # calcolate nella predizione precedente.
    stima_totale = stima_molt * contenuto_unico

    # In italiano la virgola è il separatore decimale
    molt_italiano = f"{moltiplicatore:.2f}".replace(".", ",")

    print(f"🎯 La tua stima: {stima_molt}× → ~{fmt(stima_totale)} token totali")
    print(f"✅ Risultato vero: {molt_italiano}× → {fmt(totale_processato_finale)} token totali")
    print()

    # Reazione
    opzioni_molt = [1, 2, 5, 10, 50, 100]
    opzione_piu_vicina = min(opzioni_molt, key=lambda x: abs(x - moltiplicatore))

    if stima_molt == opzione_piu_vicina:
        print("🏆 Hai indovinato la stima più vicina!")
    elif stima_molt < moltiplicatore:
        print("📈 Hai sottostimato — il modello rilegge di più di quello che pensavi.")
    else:
        print("📉 Hai sovrastimato — su 10 turni è tanto, ma non così tanto. Aspetta turno 50...")

    print()
    print(f"Il modello ha riletto i primi messaggi quasi 5 volte. È normale:")
    print("la conversazione si trascina dietro tutto, ad ogni turno, dall'inizio.")
    print()
    print(f"💸 Su API a pagamento: stai pagando {fmt(totale_processato_finale)} token per una chat")
    print(f"   di solo {fmt(contenuto_unico)} token di contenuto unico.")
    print()
    print("📈 Pensaci a turno 20: il moltiplicatore sale a ~17×. A turno 50: ~50×.")
    print("   Ecco perché chat lunghe = caro, lento, e iniziano a perdere il filo.")

    output_totale = sum(lista_token_risposta)
    print()
    print("💰 Quanto costerebbe DAVVERO questa chat di 10 turni via API?")
    print(f"   Token processati: {totale_processato_finale:,} input + {output_totale:,} output".replace(",", "."))
    print()
    for nome in PREZZI:
        c = costo(totale_processato_finale, output_totale, nome)
        print(f"   {nome:<22} ${c:.4f}")
    print()
    print("📉 Confronto: se non ci fosse il riascolto (solo 1.571 token di contenuto unico + output):")
    costo_ideale_min = costo(contenuto_unico, output_totale, "Mistral Small 3")
    costo_reale_min  = costo(totale_processato_finale, output_totale, "Mistral Small 3")
    costo_ideale_max = costo(contenuto_unico, output_totale, "Claude Opus 4.7")
    costo_reale_max  = costo(totale_processato_finale, output_totale, "Claude Opus 4.7")
    print(f"   Su Mistral Small  → pagheresti ${costo_ideale_min:.4f} invece di ${costo_reale_min:.4f}")
    print(f"   Su Claude Opus    → pagheresti ${costo_ideale_max:.4f} invece di ${costo_reale_max:.4f}")
    print()
    print("💡 A turno 50 i numeri si moltiplicano: una singola chat lunga su Claude Opus può costare quanto un caffè.")
else:
    print("🤔 Scegli una risposta dal menu sopra e riesegui ▶️")


In [ ]:
# @title 📊 Esercizio 6 · Visualizza la crescita dei token { display-mode: "form" }
# @markdown Ora che hai fatto le previsioni, vediamo il quadro completo. Premi ▶️ per il grafico della crescita dei token.

import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="talk")
palette = ["#1A1A1A", "#C7593F", "#7A2E1C"]

# Costruiamo un DataFrame in formato lungo per seaborn
df_messaggio = pd.DataFrame({
    "turno": lista_turni,
    "token": lista_token_messaggio,
    "serie": "Solo il tuo messaggio",
})
df_cumulativo = pd.DataFrame({
    "turno": lista_turni,
    "token": lista_token_cumulativi,
    "serie": "Input al turno (cumulativo)",
})
df_totale = pd.DataFrame({
    "turno": lista_turni,
    "token": lista_totale_processato,
    "serie": "Totale processato dal modello",
})
df = pd.concat([df_messaggio, df_cumulativo, df_totale], ignore_index=True)

fig, ax = plt.subplots(figsize=(12, 7))
sns.lineplot(
    data=df,
    x="turno",
    y="token",
    hue="serie",
    palette=palette,
    marker="o",
    linewidth=2.5,
    markersize=9,
    ax=ax,
)

ax.set_xlabel("Turno della chat", fontsize=14)
ax.set_ylabel("Numero di token", fontsize=14)
ax.set_title("Cosa il modello legge davvero a ogni turno",
             fontsize=18, fontweight="bold", style="italic", pad=20)
ax.set_xticks(lista_turni)
legend = ax.legend(loc="upper left", title="Cosa viene contato", frameon=True)
legend.get_title().set_fontweight("bold")
sns.despine()

# Calcoli per l'annotazione
contenuto_unico = len(enc.encode(chat_completa))
totale_letto = lista_totale_processato[-1]
moltiplicatore = totale_letto / contenuto_unico

# Formattazione italiana: punto migliaia, virgola decimale
totale_fmt = f"{totale_letto:,}".replace(",", ".")
molt_fmt = f"{moltiplicatore:.1f}".replace(".", ",")
ax.annotate(
    f"{totale_fmt} token totali\nprocessati in 10 turni",
    xy=(10, totale_letto),
    xytext=(6.5, totale_letto * 0.65),
    fontsize=13,
    color="#7A2E1C",
    fontweight="bold",
    arrowprops=dict(arrowstyle="->", color="#7A2E1C", alpha=0.6, lw=2),
)

plt.tight_layout()
plt.show()

print()
print("👀 Vedete le tre curve?")
print("   • Nera (in basso): il vostro messaggio singolo — sempre piccolo.")
print("   • Coral: l'input al turno N — cresce a ogni turno (lineare).")
print("   • Coral scuro: il TOTALE che il modello ha letto su tutti i turni — esplode (quadratica).")
print()
print(f"💡 Il modello ha riletto i primi messaggi {moltiplicatore:.2f} volte.")
print("   A turno 20 sarebbe ~17×, a turno 50 sarebbe ~50×.")
print("   Ecco perché chat lunghe = caro e lento.")
print()
print("✨ Soluzione pratica: APRI UNA CHAT NUOVA quando cambia argomento.")


---
## Bonus — Tokenizer diversi, conti diversi

Una cosa importante da sapere: **non esiste un solo modo di tokenizzare**. Ogni famiglia di modelli ha il **suo** tokenizer. La stessa frase, data a tokenizer diversi, viene spezzata in modo diverso e dà numeri diversi.

Confrontiamo tre tokenizer aperti e liberamente accessibili:
- **cl100k_base** (usato da GPT-4 / ChatGPT)
- **gpt2** (un tokenizer più vecchio, di OpenAI)
- **bert-base-multilingual-cased** (di Google, allenato su 100+ lingue)
- **xlm-roberta-base** (di Meta, multilingua)

In [ ]:
# @title 🎯 Bonus · Confronta 4 tokenizer diversi { display-mode: "form" }
# @markdown Scrivi una frase qui sotto e premi ▶️ per vedere come la spezzano 4 tokenizer diversi.

import os
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

testo_bonus = "Ciao come stai? Andiamo a Pontremoli." # @param {type:"string"}

print(f"📝 Testo: {testo_bonus}")
print("=" * 60)

risultati = []

# 1. tiktoken cl100k_base (GPT-4 / ChatGPT)
ids_gpt4 = enc.encode(testo_bonus)
n_gpt4 = len(ids_gpt4)
risultati.append(("cl100k_base (GPT-4)", n_gpt4))
print(f"🤖 GPT-4 / ChatGPT (cl100k_base):  {n_gpt4} token")

# 2. GPT-2 (tokenizer vecchio di OpenAI)
tk_gpt2 = AutoTokenizer.from_pretrained("gpt2")
ids_gpt2 = tk_gpt2.encode(testo_bonus)
n_gpt2 = len(ids_gpt2)
risultati.append(("gpt2", n_gpt2))
print(f"📜 GPT-2 (Hugging Face):           {n_gpt2} token")

# 3. BERT multilingue (Google)
tk_bert = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")
ids_bert = tk_bert.encode(testo_bonus)
n_bert = len(ids_bert)
risultati.append(("bert-base-multilingual-cased", n_bert))
print(f"🌐 BERT multilingue (Google):      {n_bert} token")

# 4. XLM-RoBERTa (Meta, multilingua)
tk_xlmr = AutoTokenizer.from_pretrained("xlm-roberta-base")
ids_xlmr = tk_xlmr.encode(testo_bonus)
n_xlmr = len(ids_xlmr)
risultati.append(("xlm-roberta-base", n_xlmr))
print(f"🔵 XLM-RoBERTa (Meta):             {n_xlmr} token")

# Tabella riassuntiva finale
print()
print(f'📊 RIASSUNTO — "{testo_bonus}"')
print()
nome_w = 36
tok_w = 5
print(f"  {'Tokenizer':<{nome_w}} | {'Token':>{tok_w}} | Differenza vs GPT-4")
print(f"  {'-' * nome_w}|{'-' * (tok_w + 2)}|--------------------")
for nome, conteggio in risultati:
    if nome == "cl100k_base (GPT-4)":
        diff_str = "(riferimento)"
    else:
        delta = conteggio - n_gpt4
        diff_str = f"{delta:+d}"
    print(f"  {nome:<{nome_w}} | {conteggio:>{tok_w}} | {diff_str}")

print()
print("💡 Lo stesso testo può valere il 50% in più o il 15% in meno")
print("   a seconda del tokenizer. Quando leggete \"max 200k token\", chiedete: di chi?")
print()
print("💡 Nota: i tokenizer di Llama (Meta) e Mistral richiedono un account")
print("   Hugging Face e accesso autorizzato. Qui usiamo tokenizer liberi,")
print("   ma il principio è identico: ogni famiglia di modelli ha il suo.")


---
## Cosa abbiamo imparato?

- **I token non sono parole.** Sono pezzettini di parole. Le parole comuni costano poco, quelle rare o lunghe costano tanto. L'italiano spesso costa più dell'inglese.

- **Ogni messaggio in una chat si "trascina dietro" tutto quello che è venuto prima.** Al turno 10, il modello legge da capo tutti i turni precedenti. Per questo le chat lunghe diventano lente, costose, e iniziano a perdere il filo.

- **Tokenizer diversi = conti diversi.** Quando leggete "max 200.000 token" o "costo 10$/M token", chiedetevi sempre di **quale** tokenizer si parla. La stessa frase può valere il doppio o la metà.

> **Trucco pratico:** quando una chat diventa lunga e iniziate a notare che il modello si "dimentica", **apri una chat nuova**. Stai risparmiando token (e quindi soldi, tempo, e qualità delle risposte).

---
## 🌍 Dove provare tutto questo

Tutto quello che abbiamo visto — token, contesto cumulativo, chat lunghe che diventano costose — vale per **ogni assistente AI** che usate. Cambia solo l'interfaccia. Qui sotto i provider più diffusi, con dove trovare le impostazioni che contano per voi: storia chat, memoria persistente, e free tier.

<table style="font-size: 1.05em; border-collapse: collapse; margin: 1em 0; width: 100%">
  <thead>
    <tr style="background: #1A1A1A; color: #F5F0E6">
      <th style="padding: 10px; text-align: left">Provider</th>
      <th style="padding: 10px; text-align: left">Storia chat</th>
      <th style="padding: 10px; text-align: left">Memoria persistente</th>
      <th style="padding: 10px; text-align: left">Editabile</th>
      <th style="padding: 10px; text-align: left">Free tier</th>
      <th style="padding: 10px; text-align: left">Search chat</th>
      <th style="padding: 10px; text-align: left">Link</th>
    </tr>
  </thead>
  <tbody>
    <tr style="background: #fdf6ee; border-bottom: 1px solid #ccc">
      <td style="padding: 8px"><b>ChatGPT</b></td>
      <td style="padding: 8px">Sì</td>
      <td style="padding: 8px">Sì, automatica</td>
      <td style="padding: 8px">Sì (Settings → Personalization → Memory)</td>
      <td style="padding: 8px">Sì</td>
      <td style="padding: 8px">Sì</td>
      <td style="padding: 8px"><a href="https://chat.openai.com">chat.openai.com</a></td>
    </tr>
    <tr style="background: #ffffff; border-bottom: 1px solid #ccc">
      <td style="padding: 8px"><b>Claude</b></td>
      <td style="padding: 8px">Sì</td>
      <td style="padding: 8px">Sì (recente) + Projects</td>
      <td style="padding: 8px">Sì</td>
      <td style="padding: 8px">Limitata</td>
      <td style="padding: 8px">Sì</td>
      <td style="padding: 8px"><a href="https://claude.ai">claude.ai</a></td>
    </tr>
    <tr style="background: #fdf6ee; border-bottom: 1px solid #ccc">
      <td style="padding: 8px"><b>Le Chat (Mistral)</b></td>
      <td style="padding: 8px">Sì</td>
      <td style="padding: 8px">Sì ("Memories")</td>
      <td style="padding: 8px">Sì</td>
      <td style="padding: 8px">Sì</td>
      <td style="padding: 8px">Sì</td>
      <td style="padding: 8px"><a href="https://chat.mistral.ai">chat.mistral.ai</a></td>
    </tr>
    <tr style="background: #ffffff; border-bottom: 1px solid #ccc">
      <td style="padding: 8px"><b>Gemini</b></td>
      <td style="padding: 8px">Sì</td>
      <td style="padding: 8px">Sì ("Saved info")</td>
      <td style="padding: 8px">Sì</td>
      <td style="padding: 8px">Sì</td>
      <td style="padding: 8px">Parziale</td>
      <td style="padding: 8px"><a href="https://gemini.google.com">gemini.google.com</a></td>
    </tr>
    <tr style="background: #fdf6ee; border-bottom: 1px solid #ccc">
      <td style="padding: 8px"><b>HuggingChat</b></td>
      <td style="padding: 8px">Sì (threads)</td>
      <td style="padding: 8px">No — ma "Assistants" custom</td>
      <td style="padding: 8px">N/A</td>
      <td style="padding: 8px">Sì (modelli open)</td>
      <td style="padding: 8px">Sì</td>
      <td style="padding: 8px"><a href="https://huggingface.co/chat">huggingface.co/chat</a></td>
    </tr>
  </tbody>
</table>

💡 *Le UI cambiano spesso. Se non trovi un'impostazione, **chiedi aiuto all'AI stessa**: "Come si modifica la memoria su [nome del servizio]?" Spesso ti guida passo per passo.*

## ⚙ Dietro le quinte

Le differenze che NON si vedono nell'interfaccia, ma che possono contare:

<table style="font-size: 1.05em; border-collapse: collapse; margin: 1em 0; width: 100%">
  <thead>
    <tr style="background: #1A1A1A; color: #F5F0E6">
      <th style="padding: 10px; text-align: left">Provider</th>
      <th style="padding: 10px; text-align: left">Azienda · Paese</th>
      <th style="padding: 10px; text-align: left">Server</th>
      <th style="padding: 10px; text-align: left">Modello aperto?</th>
      <th style="padding: 10px; text-align: left">Costo API (in/out, $/1M token)</th>
      <th style="padding: 10px; text-align: left">Usa i tuoi dati per training?</th>
    </tr>
  </thead>
  <tbody>
    <tr style="background: #fdf6ee; border-bottom: 1px solid #ccc">
      <td style="padding: 8px"><b>ChatGPT</b></td>
      <td style="padding: 8px">OpenAI · USA</td>
      <td style="padding: 8px">USA</td>
      <td style="padding: 8px">No (closed-weights)</td>
      <td style="padding: 8px">~$2,50 / ~$10 (GPT-4o)</td>
      <td style="padding: 8px">Sì di default sul tier free, opt-out in settings</td>
    </tr>
    <tr style="background: #ffffff; border-bottom: 1px solid #ccc">
      <td style="padding: 8px"><b>Claude</b></td>
      <td style="padding: 8px">Anthropic · USA</td>
      <td style="padding: 8px">USA</td>
      <td style="padding: 8px">No (closed-weights)</td>
      <td style="padding: 8px">~$3 / ~$15 (Sonnet 4.x)</td>
      <td style="padding: 8px">No di default (consumer); opt-in</td>
    </tr>
    <tr style="background: #fdf6ee; border-bottom: 1px solid #ccc">
      <td style="padding: 8px"><b>Le Chat</b></td>
      <td style="padding: 8px">Mistral · Francia 🇫🇷 (UE)</td>
      <td style="padding: 8px">UE</td>
      <td style="padding: 8px">Parzialmente (Mistral 7B, Mixtral aperti)</td>
      <td style="padding: 8px">~$0,20 / ~$0,60 (Small)</td>
      <td style="padding: 8px">Sì con opt-out</td>
    </tr>
    <tr style="background: #ffffff; border-bottom: 1px solid #ccc">
      <td style="padding: 8px"><b>Gemini</b></td>
      <td style="padding: 8px">Google · USA</td>
      <td style="padding: 8px">USA</td>
      <td style="padding: 8px">No (closed-weights)</td>
      <td style="padding: 8px">~$1,25 / ~$5 (Gemini Pro)</td>
      <td style="padding: 8px">Sì sul tier free, opt-out in Activity controls</td>
    </tr>
    <tr style="background: #fdf6ee; border-bottom: 1px solid #ccc">
      <td style="padding: 8px"><b>HuggingChat</b></td>
      <td style="padding: 8px">HuggingFace · Francia/USA</td>
      <td style="padding: 8px">Variabile (provider del modello)</td>
      <td style="padding: 8px">Sì — solo modelli aperti (Llama, Mistral, Qwen)</td>
      <td style="padding: 8px">Gratis, no API ufficiale</td>
      <td style="padding: 8px">No (solo inferenza, no salvataggio per training)</td>
    </tr>
  </tbody>
</table>

Se ti interessa la **privacy europea** → Le Chat o HuggingChat. Se vuoi un **modello aperto** che puoi anche scaricare → HuggingChat o (in parte) Le Chat. Se vuoi il **massimo della qualità** (per ora) → Claude o ChatGPT a pagamento. Per la maggior parte degli usi quotidiani, qualunque dei cinque va benissimo — la cosa importante è capire la **memoria** e il **contesto**, non il provider.